In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([54, 74, 20, 79, 32, 44, 52, 29,  5, 70, 70, 70, 43, 20, 79])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.zeros((arr.size, label), dtype=np.float32)
    one_hot[np.arange(one_hot.shape[0]), arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [5]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [6]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [7]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[54 74 20 79 32 44 52 29  5 70]
 [50 46 23 29 32 74 20 32 29 20]
 [44 23 72 29 46 52 29 20 29 75]
 [50 29 32 74 44 29 37 74 14 44]
 [29 50 20 12 29 74 44 52 29 32]
 [37 38 50 50 14 46 23 29 20 23]
 [29 24 23 23 20 29 74 20 72 29]
 [ 7 78 33 46 23 50  6 68 21 29]]
y
 [[74 20 79 32 44 52 29  5 70 70]
 [46 23 29 32 74 20 32 29 20 32]
 [23 72 29 46 52 29 20 29 75 46]
 [29 32 74 44 29 37 74 14 44 75]
 [50 20 12 29 74 44 52 29 32 44]
 [38 50 50 14 46 23 29 20 23 72]
 [24 23 23 20 29 74 20 72 29 50]
 [78 33 46 23 50  6 68 21 29  2]]


In [8]:
class CharNN(nn.Module):    
    def __init__(self, tokens, n_layer=2, n_hidden=256, drop_prob=0.5, lr=0.001):
        super().__init__()

        self.n_layer = n_layer
        self.n_hidden = n_hidden
        self.lr = lr

        self.chars = tokens
        self.int_char = dict(enumerate(self.chars))
        self.char_int = {ch : i for i, ch in self.int_char.items()}

        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layer, dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)

        self.fc = nn.Linear(n_hidden, len(self.chars))

    def forward(self, x, hidden):
        r_output, hidden = self.lstm(x, hidden)

        out = self.dropout(r_output)
       
        out = out.contiguous().view(-1, self.n_hidden)

        out = self.fc(out)

        return out, hidden
    
    def init_hidden (self, batch_size):
        
        weight = next(self.parameters()).data
        
        hidden = (weight.new(self.n_layer, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layer, batch_size, self.n_hidden).zero_())
        
        return hidden

In [9]:
def train(net, data, epoch=10, batch_size=10, seq_len=50, lr=0.01, clip=5, val_frac=0.1):

    net.train()

    opt = torch.optim.Adam(net.parameters(), lr=lr)
    crietrion = nn.CrossEntropyLoss()

    val_idx = int(len(data)*(1-val_frac))
    data, val_data = data[:val_idx], data[val_idx:]

    n_chars = len(net.chars)

    for n in range(epoch):
        h = net.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_len):
            x = one_hot_encode(x, n_chars)
            input, target = torch.from_numpy(x), torch.from_numpy(y)

            h = tuple([each.data for each in h])

            net.zero_grad()
            output, h = net(input, h)
            loss = crietrion(output, target.view(batch_size*seq_len).long())

            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        
        else:
            val_losses = []
            net.eval()
            val_h = net.init_hidden(batch_size)

            for x, y in get_batches(val_data, batch_size, seq_len):
                x = one_hot_encode(x,n_chars)
                inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

                val_h = tuple([each.data for each in val_h])

                output, val_h = net(inputs, val_h)
                val_loss = crietrion(output, targets.view(batch_size*seq_len).long())

                val_losses.append(val_loss.item())

            net.train()

            print(f"Epoch: {n+1}, Train loss: {loss.item():4f}, Val loss: {np.mean(val_losses):4f}")

In [10]:
n_hidden = 512
n_layer = 2

net = CharNN(chars, n_layer=n_layer, n_hidden=n_hidden)
print(net)

CharNN(
  (lstm): LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=83, bias=True)
)


In [ ]:
batch_size = 128
seq_len = 100
epoch = 10

train(net, encoded, epoch, batch_size, seq_len)

Epoch: 1, Train loss: 3.097653, Val loss: 3.111702
Epoch: 2, Train loss: 2.749155, Val loss: 2.672285
Epoch: 3, Train loss: 2.510003, Val loss: 2.415669
Epoch: 4, Train loss: 2.386825, Val loss: 2.261676
Epoch: 5, Train loss: 2.310861, Val loss: 2.169334


In [ ]:
model_name = 'CheckPoint_10.pth'

checkpoint = {
    'n_hidden': net.n_hidden,
    'n_layer': net.n_layer,
    'state_dict': net.state_dict(),
    'chars': net.chars
}

with open(model_name, 'wb') as f:
    torch.save(checkpoint, f)